In [ ]:
# @title ⚙️ Setup — run this first to load images
import sys

if 'google.colab' in sys.modules:
    from IPython.display import display, Javascript
    js = """
(function() {
  // Auto-detect GitHub repo from the Colab URL
  // e.g. colab.research.google.com/github/USER/REPO/blob/BRANCH/notebook.ipynb
  var m = window.location.href.match(
    /colab\.research\.google\.com\/github\/([^\/]+\/[^\/]+)\/blob\/([^\/]+)/
  );
  if (!m) { console.warn('Not opened from GitHub — images may not show.'); return; }
  var rawBase = 'https://raw.githubusercontent.com/' + m[1] + '/' + m[2];

  function patch() {
    document.querySelectorAll('img[src^="images/"]').forEach(function(img) {
      img.src = rawBase + '/' + img.getAttribute('src');
    });
  }
  patch();
  new MutationObserver(patch).observe(document.body, {childList:true, subtree:true});
  [300,800,1500,3000].forEach(function(d){ setTimeout(patch,d); });
  console.log('✅ Image patcher active →', rawBase);
})();
""";
    display(Javascript(js))
    print('✅ Images will load from your GitHub repo automatically.')
else:
    print('✅ Local mode — open with Jupyter Lab or Jupyter Notebook for images to display.')


# Part 2: Supervised Learning - Regression

**Quick Reference Guide for Regression Models**

[Back to Index](Index.ipynb)

---
## 2.1 Simple Linear Regression

**Formula:** `y = β₀ + β₁x`

**Use Case:** Single independent variable predicting target

**Assumptions:** Linearity, independence, homoscedasticity, normality

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

# Prepare data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model parameters
print(f"Slope (β₁): {model.coef_[0]}")
print(f"Intercept (β₀): {model.intercept_}")
print(f"R² Score: {model.score(X_test, y_test):.3f}")

# Visualize
plt.scatter(X_test, y_test, color='blue', label='Actual')
plt.plot(X_test, y_pred, color='red', label='Predicted')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

---
## 2.2 Multiple Linear Regression

**Formula:** `y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ`

**Use Case:** Multiple independent variables predicting target

In [ ]:
# Same as Simple Linear Regression but with multiple features
model = LinearRegression()
model.fit(X_train, y_train)  # X_train has multiple columns

# Coefficients for each feature
coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
print(coefficients.sort_values('Coefficient', ascending=False))

# Predict
y_pred = model.predict(X_test)
print(f"R² Score: {model.score(X_test, y_test):.3f}")

---
## 2.3 Polynomial Regression

**Formula:** `y = β₀ + β₁x + β₂x² + β₃x³ + ...`

**Use Case:** Non-linear relationships

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Create polynomial features
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly_train = poly.fit_transform(X_train)
X_poly_test = poly.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_poly_train, y_train)

# Predict
y_pred = model.predict(X_poly_test)
print(f"R² Score: {model.score(X_poly_test, y_test):.3f}")

# Visualize (for single feature)
X_range = np.linspace(X_train.min(), X_train.max(), 300).reshape(-1, 1)
X_range_poly = poly.transform(X_range)
y_range = model.predict(X_range_poly)

plt.scatter(X_train, y_train, color='blue', label='Training')
plt.scatter(X_test, y_test, color='green', label='Test')
plt.plot(X_range, y_range, color='red', label='Polynomial Fit')
plt.legend()
plt.show()

---
## 2.4 Support Vector Regression (SVR)

**Concept:** Finds hyperplane with maximum margin, allows errors within epsilon tube

**Use Case:** Non-linear patterns, robust to outliers

**Kernels:** 'linear', 'poly', 'rbf' (default), 'sigmoid'

### 🧠 Support Vector Regression (SVR) — Intuition + Working

SVR is the **regression version of Support Vector Machines (SVM)**.
Instead of classifying points, it tries to **fit a function that stays within a margin (ε)** from actual values.

---

### 📊 Intuition (Core Idea)

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell8_img0_bd3406ce.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell8_img1_19cf19d7.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell8_img2_724a21ca.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell8_img3_3001ece8.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell8_img4_0e6030b7.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell8_img5_1c8e4c35.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

👉 Think like this:

* You draw a **line (or curve)** that fits the data

* But instead of minimizing exact error, SVR says:

  👉 “I’m okay if predictions are within ±ε (epsilon) range”

* Only points **outside this margin** matter → called **support vectors**

---

### 📌 Key Concept: ε-Insensitive Tube

SVR builds a **tube around the regression line**

* Errors **inside the tube (±ε)** → ignored ✅
* Errors **outside the tube** → penalized ❌

---

### 🔢 Mathematical Objective

SVR tries to:

$$
\min \frac{1}{2}\|w\|^2 + C \sum (\xi_i + \xi_i^*)
$$

Subject to:
$$
|y_i - f(x_i)| \leq \epsilon
$$
---

### 🧩 Parameters (VERY IMPORTANT)

#### 🔹 1. ε (epsilon)

* Defines **margin width**
* Larger ε → more tolerance → smoother model

---

#### 🔹 2. C (regularization)

* Controls penalty for errors
* High C → tries to fit all points (overfitting risk)
* Low C → allows more errors (generalization)

---

#### 🔹 3. Kernel

Transforms data to higher dimension

Common kernels:

* Linear
* Polynomial
* RBF (most used 🔥)

---

### 📊 Example

| x | y            |
| - | ------------ |
| 1 | 2            |
| 2 | 2.5          |
| 3 | 3            |
| 4 | 10 (outlier) |

👉 SVR will:

* Ignore small deviations
* Focus on major violations (like 10)

---

### 🚀 Python Example (sklearn)

```python
from sklearn.svm import SVR
import numpy as np

X = np.array([[1], [2], [3], [4]])
y = np.array([2, 2.5, 3, 10])

model = SVR(kernel='rbf', C=100, epsilon=0.1)
model.fit(X, y)

pred = model.predict(X)
print(pred)
```

---

### ⚔️ SVR vs Linear Regression

| Feature            | SVR            | Linear Regression   |
| ------------------ | -------------- | ------------------- |
| Error handling     | ε-insensitive  | minimize all errors |
| Robust to outliers | ✅ Yes          | ❌ No                |
| Flexibility        | High (kernels) | Low                 |
| Complexity         | High           | Low                 |

---

### 🧠 Interview Insight (VERY IMPORTANT)

👉 If asked:

**“Why SVR instead of Linear Regression?”**

Answer:

* Handles **non-linear relationships (via kernels)**
* Ignores small noise (ε margin)
* More robust to outliers

---

### ⚠️ Limitations

* Slow for large datasets ❌
* Hard to tune (C, ε, kernel)
* Not easily interpretable

---

### 🚀 Real-world Use Cases

* Stock price prediction
* Demand forecasting
* Time-series regression (small-medium data)

---

### 💡 One-liner to Remember

👉 **SVR = Regression with margin tolerance (ε) using support vectors**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# SVR requires feature scaling!
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

# Train SVR
model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
# C: regularization (smaller = more regularization)
# epsilon: width of epsilon tube
model.fit(X_train_scaled, y_train_scaled)

# Predict and inverse transform
y_pred_scaled = model.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))

# Evaluate
from sklearn.metrics import r2_score
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")

---
## 2.5 Decision Tree Regression

**Concept:** Splits data based on features to minimize variance

**Pros:** Non-linear, no scaling needed, interpretable

**Cons:** Prone to overfitting

### 🌳 Decision Tree Regression — Clear Explanation

Decision Tree Regression is a **non-linear model** that predicts values by **splitting data into regions** and assigning a constant value (usually mean) in each region.

---

### 📊 Intuition

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell11_img0_b49a45b8.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell11_img1_4d9cf647.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell11_img2_6b9b30d8.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell11_img3_0877342d.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell11_img4_7741c954.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell11_img5_1e39e70a.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell11_img6_72ca85df.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

👉 Think like this:

* Instead of fitting one line (like linear regression),
* It **splits the data step-by-step** based on feature values
* Each split creates a **region**, and each region predicts a **constant value**

---

### 🧠 How It Works (Step-by-Step)

1. Start with all data
2. Try different splits (e.g., `x < 5`)
3. Choose split that **minimizes error (MSE)**
4. Repeat recursively
5. Stop based on conditions (depth, min samples)

---

### 🔢 Objective Function

Decision tree minimizes **Mean Squared Error (MSE)**:

$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

👉 At each split, it picks the feature + threshold that reduces MSE the most

---

### 📊 Example

#### Input

| Size (sqft) | Price |
| ----------- | ----- |
| 500         | 50    |
| 800         | 80    |
| 1200        | 150   |
| 1500        | 200   |

---

#### Tree Logic

```
if size < 1000:
    predict ≈ 65
else:
    predict ≈ 175
```

👉 Predictions are **step-like (piecewise constant)**

---

### 🚀 Python Example

```python
from sklearn.tree import DecisionTreeRegressor
import numpy as np

X = np.array([[500], [800], [1200], [1500]])
y = np.array([50, 80, 150, 200])

model = DecisionTreeRegressor(max_depth=2)
model.fit(X, y)

pred = model.predict(X)
print(pred)
```

---

### ⚙️ Important Parameters

#### 🔹 max_depth

* Controls tree height
* Too high → overfitting

---

#### 🔹 min_samples_split

* Minimum samples to split a node

---

#### 🔹 min_samples_leaf

* Minimum samples in a leaf node

---

#### 🔹 max_features

* Number of features considered per split

---

### ⚔️ Decision Tree vs Linear Regression

| Feature          | Decision Tree | Linear Regression |
| ---------------- | ------------- | ----------------- |
| Relationship     | Non-linear ✅  | Linear            |
| Interpretability | Medium        | High              |
| Handles outliers | Better        | Poor              |
| Overfitting risk | High ❌        | Low               |

---

### 🔥 Advantages

* Captures **non-linear patterns**
* No need for feature scaling
* Easy to interpret (rules)

---

### ⚠️ Disadvantages

* Overfitting (very common ❌)
* Unstable (small data change → big tree change)
* Piecewise constant (not smooth)

---

### 🚀 Real-world Use Cases

* Price prediction (house, product)
* Demand forecasting
* Feature engineering (tree-based models)

---

### 🧠 Interview Insight (VERY IMPORTANT)

👉 If interviewer asks:

**“Why trees instead of linear models?”**

Answer:

* Capture **non-linear relationships**
* No assumptions about data distribution
* Handle interactions automatically

---

### ⚡ Pro Tip (MLE Level)

👉 Decision Trees are rarely used alone in production
Instead use:

* Random Forest 🌲🌲
* Gradient Boosting (XGBoost, LightGBM) 🔥

---

### 💡 One-liner to Remember

👉 **Decision Tree Regression = Split data into regions → predict mean per region**


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree

# Train model
model = DecisionTreeRegressor(
    max_depth=5,           # limit tree depth to prevent overfitting
    min_samples_split=20,  # minimum samples to split node
    min_samples_leaf=10,   # minimum samples in leaf
    random_state=42
)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
print(f"R² Score: {model.score(X_test, y_test):.3f}")

# Feature importance
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importances)

# Visualize tree (small trees only)
plt.figure(figsize=(20, 10))
tree.plot_tree(model, feature_names=X.columns, filled=True, max_depth=3)
plt.show()

---
## 2.6 Random Forest Regression

**Concept:** Ensemble of decision trees (bagging)

**Pros:** Reduces overfitting, handles non-linearity, robust

**Use Case:** Most versatile regression algorithm

### 🌲 Random Forest — Intuition, Working & Why It’s Powerful

Random Forest is an **ensemble model** that builds **many decision trees** and combines their predictions to get a **more accurate and stable result**.

---

### 📊 Intuition

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell14_img0_4c6d00fc.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell14_img1_4d466c65.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell14_img2_0956d665.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell14_img3_e1fca0e5.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell14_img4_298f44ac.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

👉 Think like this:

* Instead of trusting **one decision tree (which overfits)**
* You train **many trees on different subsets of data**
* Then:

  * Regression → **average predictions**
  * Classification → **majority voting**

---

### 🧠 How It Works (Step-by-Step)

#### 1. Bootstrap Sampling (Bagging)

* Randomly sample data **with replacement**
* Each tree sees a **different dataset**

---

#### 2. Random Feature Selection

* At each split, use only a **subset of features**
* Reduces correlation between trees

---

#### 3. Train Multiple Trees

* Build many decision trees independently

---

#### 4. Aggregate Results

* Regression:
$$
\hat{y}(x) = \frac{1}{N} \sum_{t=1}^{N} \hat{y}_t(x)
$$
* Classification:
  → Majority vote

---

### 📊 Example

Suppose 3 trees predict:

| Tree | Prediction |
| ---- | ---------- |
| T1   | 100        |
| T2   | 110        |
| T3   | 90         |

👉 Final prediction:

```text
(100 + 110 + 90) / 3 = 100
```

---

### ⚙️ Important Parameters

#### 🔹 n_estimators

* Number of trees
* More trees → better performance (but slower)

---

#### 🔹 max_depth

* Depth of each tree
* Controls overfitting

---

#### 🔹 max_features

* Number of features per split
* Key for randomness

---

#### 🔹 min_samples_leaf

* Minimum samples in leaf node

---

### ⚔️ Random Forest vs Decision Tree

| Feature          | Decision Tree | Random Forest |
| ---------------- | ------------- | ------------- |
| Overfitting      | High ❌        | Low ✅         |
| Stability        | Low           | High          |
| Accuracy         | Medium        | High          |
| Interpretability | Easy          | Hard          |

---

### 🔥 Why Random Forest Works

👉 Reduces **variance** (main problem of trees)

* Individual trees → high variance
* Averaging → cancels noise

---

### 🚀 Python Example (sklearn)

```python
from sklearn.ensemble import RandomForestRegressor
import numpy as np

X = np.array([[500], [800], [1200], [1500]])
y = np.array([50, 80, 150, 200])

model = RandomForestRegressor(n_estimators=100, max_depth=3)
model.fit(X, y)

pred = model.predict(X)
print(pred)
```

---

### 🧠 Interview Insight (VERY IMPORTANT)

👉 If interviewer asks:

**“Why Random Forest over Decision Tree?”**

Answer:

* Reduces overfitting
* Better generalization
* Handles noise well

---

### ⚠️ Limitations

* Slower than single tree
* Less interpretable
* Large memory usage

---

### 🚀 Real-world Use Cases

* Credit scoring
* Fraud detection
* Recommendation systems
* Feature importance extraction

---

### 🔥 Bonus: Feature Importance

```python
model.feature_importances_
```

👉 Helps in **feature selection**

---

### ⚡ Random Forest vs XGBoost (Quick Insight)

| Feature  | Random Forest | XGBoost    |
| -------- | ------------- | ---------- |
| Training | Parallel      | Sequential |
| Bias     | Higher        | Lower      |
| Speed    | Faster        | Slower     |
| Accuracy | Good          | Better 🔥  |

---

### 💡 One-liner to Remember

👉 **Random Forest = Many decision trees + averaging = better accuracy & less overfitting**


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train model
model = RandomForestRegressor(
    n_estimators=100,      # number of trees
    max_depth=10,          # max depth of each tree
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',   # features to consider for split
    random_state=42,
    n_jobs=-1              # use all CPU cores
)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
print(f"R² Score: {model.score(X_test, y_test):.3f}")

# Feature importance
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(importances['feature'][:10], importances['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.show()

---
## 2.7 Bias Variance Tradeoff

**Reference:** https://mlu-explain.github.io/bias-variance/

### 🎯 Bias–Variance Tradeoff (with Technical Intuition)

At a high level, prediction error can be decomposed as:

$$
\mathbb{E}[(y - \hat{f}(x))^2] = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}
$$

* **Bias** → error from wrong assumptions (underfitting)
* **Variance** → error from sensitivity to data (overfitting)
* **Irreducible error** → noise you can’t remove

---

### 🧠 Intuition

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell16_img0_2fee433b.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell16_img1_aeb772fd.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell16_img2_1b6ec91f.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell16_img3_ce918cf0.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell16_img4_086b4b50.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="images/Part2_Supervised_Regression_cell16_img5_ac4a0733.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

* **Simple model** → high bias, low variance
* **Complex model** → low bias, high variance
* Sweet spot → **balanced tradeoff**

---

### 📊 Technical Example (Regression)

Let’s take a true function:

```python
y = sin(x) + noise
```

We try 3 models:

---

### 🔴 Model 1: Linear Regression (Underfitting)

```python
y = ax + b
```

* Cannot capture sine curve
* Predictions always off

👉 **High Bias, Low Variance**

---

### 🟡 Model 2: Polynomial (degree = 4)

* Captures curve well
* Generalizes properly

👉 **Balanced (Optimal)**

---

### 🔵 Model 3: Polynomial (degree = 15)

* Fits training data perfectly
* But learns noise too

👉 **Low Bias, High Variance**

---

### 📉 Error Behavior

| Model   | Train Error | Test Error |
| ------- | ----------- | ---------- |
| Simple  | High        | High       |
| Optimal | Low         | Low        |
| Complex | Very Low    | High       |

---

### 🧪 Code Example (sklearn)

```python
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

### Data
X = np.linspace(0, 2*np.pi, 100).reshape(-1, 1)
y = np.sin(X).ravel() + np.random.normal(0, 0.1, 100)

### Models
models = [
    make_pipeline(PolynomialFeatures(1), LinearRegression()),   # high bias
    make_pipeline(PolynomialFeatures(4), LinearRegression()),   # optimal
    make_pipeline(PolynomialFeatures(15), LinearRegression())   # high variance
]

for model in models:
    model.fit(X, y)
```

---

### ⚔️ Bias vs Variance

| Feature | Bias                     | Variance            |
| ------- | ------------------------ | ------------------- |
| Meaning | Wrong assumptions        | Sensitivity to data |
| Problem | Underfitting             | Overfitting         |
| Example | Linear on nonlinear data | Deep tree           |
| Fix     | Increase complexity      | Regularization      |

---

### 🔥 Real Model Examples

| Model             | Bias   | Variance |
| ----------------- | ------ | -------- |
| Linear Regression | High   | Low      |
| Decision Tree     | Low    | High     |
| Random Forest     | Medium | Medium   |
| XGBoost           | Low    | Medium   |

---

### ⚙️ How to Control Tradeoff

### Reduce Bias (Underfitting)

* Increase model complexity
* Add features
* Use non-linear models

---

### Reduce Variance (Overfitting)

* Regularization (L1/L2)
* Reduce depth (trees)
* More data
* Drop features

---

### 🧠 Interview Insight (VERY IMPORTANT)

👉 If asked:

**“Why Random Forest works?”**

Answer:

* Decision trees → high variance
* Random Forest → averages trees → reduces variance

---

### ⚡ Practical MLE Insight

In production:

* Start simple → increase complexity
* Use cross-validation
* Monitor **train vs validation gap**

---

### 💡 One-liner to Remember

👉 **Bias = underfitting, Variance = overfitting, goal = balance both**

---

---
## 2.7 Advanced Regression Techniques

### Ridge Regression (L2 Regularization)

**Formula:** `Cost = MSE + α * Σ(β²)`

**Use Case:** Prevent overfitting, multicollinearity

**Effect:** Shrinks coefficients but doesn't set them to zero

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV

# Ridge with fixed alpha
model = Ridge(alpha=1.0)  # higher alpha = more regularization
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Ridge with cross-validated alpha selection
model_cv = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
model_cv.fit(X_train, y_train)
print(f"Best alpha: {model_cv.alpha_}")
print(f"R² Score: {model_cv.score(X_test, y_test):.3f}")

### Lasso Regression (L1 Regularization)

**Formula:** `Cost = MSE + α * Σ|β|`

**Use Case:** Feature selection, sparse models

**Effect:** Can set coefficients to exactly zero (feature selection)

In [ ]:
from sklearn.linear_model import Lasso, LassoCV

# Lasso with fixed alpha
model = Lasso(alpha=0.1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Lasso with CV
model_cv = LassoCV(alphas=[0.001, 0.01, 0.1, 1.0], cv=5)
model_cv.fit(X_train, y_train)
print(f"Best alpha: {model_cv.alpha_}")
print(f"R² Score: {model_cv.score(X_test, y_test):.3f}")

# Features selected (non-zero coefficients)
selected = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model_cv.coef_
})
print(f"\nFeatures with non-zero coefficients:")
print(selected[selected['coefficient'] != 0].sort_values('coefficient', key=abs, ascending=False))

### Elastic Net (L1 + L2 Regularization)

**Formula:** `Cost = MSE + α * (l1_ratio * Σ|β| + (1-l1_ratio) * Σ(β²))`

**Use Case:** Combines benefits of Ridge and Lasso

In [ ]:
from sklearn.linear_model import ElasticNet, ElasticNetCV

# ElasticNet
model = ElasticNet(alpha=0.1, l1_ratio=0.5)  # l1_ratio: 0=Ridge, 1=Lasso
model.fit(X_train, y_train)

# ElasticNet with CV
model_cv = ElasticNetCV(alphas=[0.1, 1.0, 10.0], l1_ratio=[0.1, 0.5, 0.9], cv=5)
model_cv.fit(X_train, y_train)
print(f"Best alpha: {model_cv.alpha_}")
print(f"Best l1_ratio: {model_cv.l1_ratio_}")
print(f"R² Score: {model_cv.score(X_test, y_test):.3f}")

### Huber Regression

**Use Case:** Robust to outliers

**Effect:** Combines squared loss for small errors, absolute loss for large errors

In [ ]:
from sklearn.linear_model import HuberRegressor

model = HuberRegressor(epsilon=1.35, alpha=0.0)  # epsilon: threshold for outliers
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"R² Score: {model.score(X_test, y_test):.3f}")

**Regularization Comparison:**

| Method | Regularization | Feature Selection | Use Case |
|--------|---------------|-------------------|----------|
| Ridge | L2 (β²) | No | Multicollinearity |
| Lasso | L1 (\|β\|) | Yes | Sparse models |
| ElasticNet | L1 + L2 | Yes | Best of both |
| Huber | - | No | Outliers |

---
## 2.8 Regression Model Evaluation

**Key Metrics for Regression**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Assuming y_test and y_pred are available

# 1. Mean Absolute Error (MAE)
# Average absolute difference
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.3f}")

# 2. Mean Squared Error (MSE)
# Average squared difference (penalizes large errors more)
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.3f}")

# 3. Root Mean Squared Error (RMSE)
# Square root of MSE (same unit as target)
rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.3f}")

# 4. R-squared (R²)
# Proportion of variance explained (0 to 1, higher is better)
r2 = r2_score(y_test, y_pred)
print(f"R²: {r2:.3f}")

# 5. Adjusted R²
# Adjusts R² for number of features
n = len(y_test)
p = X_test.shape[1]  # number of features
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print(f"Adjusted R²: {adj_r2:.3f}")

# 6. Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
print(f"MAPE: {mape:.2f}%")

**Metric Comparison:**

| Metric | Range | Outlier Sensitive | Interpretation |
|--------|-------|-------------------|----------------|
| MAE | 0 to ∞ | No | Average error magnitude |
| MSE | 0 to ∞ | Yes | Squared error (penalizes large errors) |
| RMSE | 0 to ∞ | Yes | Error in original units |
| R² | -∞ to 1 | Yes | Variance explained (1 = perfect) |
| Adj R² | -∞ to 1 | Yes | R² adjusted for features |
| MAPE | 0 to ∞ | No | Error as percentage |

### 📊 R² and Adjusted R² — Clear + Technical Explanation

---

### 🧠 1. R² (R-squared)

👉 **R² measures how well your model explains the variance in the data**

---

### 🔢 Formula

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

Where:

* $SS_{res}$ = residual sum of squares (model error)
* $SS_{tot}$ = total variance in data

---

### 📌 Interpretation

| R² Value | Meaning              |
| -------- | -------------------- |
| 1        | Perfect fit ✅        |
| 0        | No explanatory power |
| < 0      | Worse than mean ❌    |

---

### 📊 Example

If:

* R² = 0.8

👉 Model explains **80% of variance** in target

---

### ⚠️ Problem with R²

👉 R² **always increases** when you add more features
Even if those features are useless ❌

---

### 🧠 2. Adjusted R²

👉 Fixes the problem of R²

👉 Penalizes adding unnecessary features

---

### 🔢 Formula

$$
\text{Adjusted } R^2 = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}
$$

Where:

* $n$ = number of samples
* $p$ = number of features

---

### 📌 Key Idea

* Adds penalty for more features
* Only increases if feature **actually improves model**

---

### 📊 Example

| Model   | Features | R²   | Adjusted R² |
| ------- | -------- | ---- | ----------- |
| Model A | 2        | 0.80 | 0.78        |
| Model B | 10       | 0.85 | 0.76        |

👉 Even though R² ↑, Adjusted R² ↓
➡️ Extra features are useless

---

### ⚔️ R² vs Adjusted R²

| Feature            | R²               | Adjusted R²       |
| ------------------ | ---------------- | ----------------- |
| Penalizes features | ❌ No             | ✅ Yes             |
| Always increases   | ✅ Yes            | ❌ No              |
| Use case           | Basic evaluation | Feature selection |

---

### 🧠 Intuition

* **R²**: “How well do I fit the data?”
* **Adjusted R²**: “Am I overfitting while doing it?”

---

### 🚀 Python Example

```python
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

X = np.array([[1], [2], [3], [4]])
y = np.array([2, 4, 6, 8])

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

r2 = r2_score(y, y_pred)

n = len(y)
p = X.shape[1]

adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("R2:", r2)
print("Adjusted R2:", adj_r2)
```

---

### 🧠 Interview Insight (VERY IMPORTANT)

👉 If interviewer asks:

**“Why Adjusted R²?”**

Answer:

* R² can be misleading
* Adjusted R² prevents overfitting by penalizing extra features

---

### ⚡ Practical Insight (MLE Level)

* Use:

  * R² → quick evaluation
  * Adjusted R² → feature selection

* But in real ML:

  * Prefer **cross-validation metrics** (RMSE, MAE)

---

### 💡 One-liner to Remember

👉 **R² = goodness of fit**
👉 **Adjusted R² = goodness of fit with penalty for extra features**

---

#### Cross-Validation for Regression

In [ ]:
from sklearn.model_selection import cross_val_score, cross_validate

# K-Fold Cross-Validation
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Single metric
scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f"R² scores: {scores}")
print(f"Mean R²: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")

# Multiple metrics
scoring = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
scores = cross_validate(model, X, y, cv=5, scoring=scoring)

print(f"\nR²: {scores['test_r2'].mean():.3f}")
print(f"MSE: {-scores['test_neg_mean_squared_error'].mean():.3f}")
print(f"MAE: {-scores['test_neg_mean_absolute_error'].mean():.3f}")

### Residual Analysis

In [ ]:
# Calculate residuals
residuals = y_test - y_pred

# 1. Residual Plot (check for patterns)
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()
# Good: Random scatter around zero
# Bad: Pattern (non-linearity, heteroscedasticity)

# 2. Histogram of Residuals (check normality)
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=30, edgecolor='black')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals')
plt.show()
# Good: Normal distribution (bell curve)

# 3. Q-Q Plot (check normality)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot')
plt.show()
# Good: Points close to diagonal line

# 4. Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted')
plt.show()
# Good: Points close to diagonal line

---
### Model Comparison

| Model | Linearity | Scaling Required | Overfitting Risk | Interpretability |
|-------|-----------|------------------|------------------|------------------|
| Linear Regression | Linear | No | Low | High |
| Polynomial Regression | Non-linear | No | High | Medium |
| SVR | Non-linear | Yes | Medium | Low |
| Decision Tree | Non-linear | No | High | High |
| Random Forest | Non-linear | No | Low | Medium |
| Ridge/Lasso | Linear | No | Low | High |

**Quick Decision Guide:**
- Linear relationship → Linear/Ridge/Lasso
- Non-linear → Polynomial/SVR/Random Forest
- Many features → Lasso/ElasticNet (feature selection)
- Outliers → Huber/Random Forest
- Need interpretability → Linear/Decision Tree
- Best overall performance → Random Forest/Gradient Boosting